# A/B: chunk 2 glitch — proses baru dan konteks 30 detik

Dua uji memakai bobot yang sudah ada, tanpa generasi ulang seluruh lagu:

1. **Input chunk 2 yang sama persis**, seed dan parameter sama, tetapi proses
   Python/model baru. Membantu memeriksa efek pemakaian ulang handler atau variasi
   inferensi. Hasil MPS tidak dijamin bit-identical.
2. **30 detik pertama dalam satu inferensi tanpa stitching.** Menguji apakah
   konteks lebih panjang memberi bass yang lebih konsisten. Model masih Base,
   50 langkah, CFG 7, planner mati; tidak ada metadata baru atau normalisasi kedua.

Kedua uji berjalan saat Run All. Tes 30 detik dapat membutuhkan memori/waktu lebih
besar daripada 10 detik, dan belum dibenchmark pada Mac 24 GB. Ini belum uji
180 detik sekaligus. Error OOM disimpan oleh runner; hasil sebelumnya tetap ada.

Notebook tidak menjanjikan glitch teratasi. Dengarkan perbandingan sebelum
menentukan apakah harus mengubah state model, konteks, atau konfigurasi generasi.


In [ ]:
from pathlib import Path
import json, sys, uuid
from IPython.display import Audio, display
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts"))
sys.path.insert(0, str(ROOT / "tools/stem_eval"))
from context_comparison import prepare_comparison
from mac_workflow import run_command, check_model_runtime
from setup_runtime import runtime_python
from stream_process import run_logged
from seam_diagnostics import signal_stats
from audio import read_pcm
RUN = None  # atau Path folder run notebook 03 yang bermasalah
DRY_RUN = False
if RUN is None:
    candidates = sorted((ROOT / "runs").glob("full_song_*/experiment.json"), key=lambda p: p.stat().st_mtime, reverse=True)
    candidates = [p for p in candidates if json.loads(p.read_text()).get("status") == "generated"]
    assert candidates, "Pilih RUN dari hasil lagu penuh yang sudah selesai"
    RUN = candidates[0].parent
WORK = ROOT / "runs" / ("context_comparison_" + uuid.uuid4().hex[:12])
comparison = prepare_comparison(Path(RUN), WORK, chunk_index=2)
PROFILE = comparison["profile"]
print(json.dumps(comparison, indent=2))
print("Hasil pembanding:", WORK)
display(Audio(filename=str(WORK / "original_chunk_stem.wav")))


## 1. Ulang chunk 2 dengan input/seed yang sama, proses baru

Input disalin utuh dari cache prepared. Gain 0.7 pada sumber tidak diterapkan lagi.
Output selalu ke folder baru, sehingga hasil buruk lama tidak menimpa pembanding.


In [ ]:
FRESH_RESULT = WORK / "fresh_result"
if DRY_RUN:
    run_logged(run_command(ROOT, PROFILE, Path(comparison["fresh_chunk"]), WORK / "fresh_plan"), check=True)
    print("DRY RUN: belum ada inferensi model")
else:
    print(json.dumps(check_model_runtime(runtime_python()), indent=2))
    run_logged(run_command(ROOT, PROFILE, Path(comparison["fresh_chunk"]), FRESH_RESULT, execute=True), check=True)


## 2. Satu generasi 30 detik, tanpa sambungan antar-chunk

Pengambilan sumber dibatasi 30 detik, dengan panjang lebih pendek bila lagu tidak
mencapai 30 detik. Gunakan sel ini setelah mendengarkan hasil uji proses baru jika
ingin menilai kebutuhan eksperimen berikutnya satu per satu.


In [ ]:
CONTEXT_RESULT = WORK / "context_result"
if DRY_RUN:
    command = run_command(ROOT, PROFILE, Path(comparison["context"]), WORK / "context_plan")
else:
    command = run_command(ROOT, PROFILE, Path(comparison["context"]), CONTEXT_RESULT, execute=True)
command += ["--max-job-seconds", "30"]
run_logged(command, check=True)


## 3. Bandingkan

- Lama glitch, fresh chunk bersih: indikasi state/cache handler atau variasi
  inferensi; belum membuktikan satu bug tertentu.
- Fresh chunk juga glitch, konteks 30 detik bersih: indikasi konteks pendek/input
  lokal berperan. Uji lebih panjang bertahap, jangan menganggap 3 menit pasti muat.
- Keduanya glitch: periksa sumber, seed/parameter, decoder dan clipping; menambah
  crossfade tidak memperbaiki audio yang sudah rusak sebelum stitching.
- Perbedaan karakter bass antar-generasi independen tetap mungkin meski bebas glitch.

Laporkan yang terdengar bersih/rusak dan kirim cuplikan pembanding jika bersedia.
Statistik sinyal di bawah tidak otomatis menilai kualitas musik.


In [ ]:
print("LAMA: chunk 2 sebelum stitching")
display(Audio(filename=str(WORK / "original_chunk_stem.wav")))
for label, folder in (("FRESH: chunk sama, proses baru", FRESH_RESULT), ("CONTEXT: satu inferensi hingga 30 detik", CONTEXT_RESULT)):
    results = sorted(folder.glob("*/stem.wav")) if folder.exists() else []
    for path in results:
        print(label)
        print(json.dumps(signal_stats(read_pcm(path)), indent=2))
        display(Audio(filename=str(path)))
        display(Audio(filename=str(path.parent / "mix.wav")))
if DRY_RUN:
    print("Hanya plan; tidak ada hasil model baru")
